For this module, we'll use Isolation Forest, an unsupervised algorithm. It can flag unusual machine operating conditions even when we don't have a specific fault label.


Imports + load AI4I


In [2]:
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

df = pd.read_csv("../data/raw/ai4i2020.csv")

print("Dataset shape:", df.shape)
print("\nMissing values:", df.isnull().sum().sum())
print("\nMachine failure distribution:")
print(df["Machine failure"].value_counts())

df.head()

Dataset shape: (10000, 14)

Missing values: 0

Machine failure distribution:
Machine failure
0    9661
1     339
Name: count, dtype: int64


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


Select features

In [3]:
anomaly_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

X_anomaly = df[anomaly_features].copy()

print("Feature shape:", X_anomaly.shape)
print("\nMissing values:")
print(X_anomaly.isnull().sum())

Feature shape: (10000, 5)

Missing values:
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
dtype: int64


Scale the features

In [4]:
scaler = StandardScaler()

X_anomaly_scaled = scaler.fit_transform(X_anomaly)

print("Scaled shape:", X_anomaly_scaled.shape)

print(
    "Any NaN:",
    np.isnan(X_anomaly_scaled).any()
)

Scaled shape: (10000, 5)
Any NaN: False


Train Isolation Forest

In [5]:
isolation_model = IsolationForest(
    n_estimators=300,
    contamination=0.04,
    random_state=42,
    n_jobs=-1
)

isolation_model.fit(X_anomaly_scaled)

print("Isolation Forest trained successfully!")

Isolation Forest trained successfully!


Detect anomalies

In [6]:
raw_predictions = isolation_model.predict(
    X_anomaly_scaled
)

df["Anomaly"] = np.where(
    raw_predictions == -1,
    1,
    0
)

df["Anomaly Score"] = (
    -isolation_model.decision_function(
        X_anomaly_scaled
    )
)

print("Anomaly distribution:")
print(df["Anomaly"].value_counts())

print(
    "\nAnomaly percentage:",
    round(df["Anomaly"].mean() * 100, 2),
    "%"
)

Anomaly distribution:
Anomaly
0    9600
1     400
Name: count, dtype: int64

Anomaly percentage: 4.0 %


Compare anomalies with real failures

In [7]:
comparison = pd.crosstab(
    df["Machine failure"],
    df["Anomaly"],
    rownames=["Actual Machine Failure"],
    colnames=["Detected Anomaly"]
)

comparison

Detected Anomaly,0,1
Actual Machine Failure,,
0,9322,339
1,278,61


In [8]:
print(
    classification_report(
        df["Machine failure"],
        df["Anomaly"],
        target_names=["Normal", "Failure"],
        zero_division=0
    )
)

              precision    recall  f1-score   support

      Normal       0.97      0.96      0.97      9661
     Failure       0.15      0.18      0.17       339

    accuracy                           0.94     10000
   macro avg       0.56      0.57      0.57     10000
weighted avg       0.94      0.94      0.94     10000



IMPORTANT RESULTS

In [9]:
total_failures = (
    df["Machine failure"] == 1
).sum()

captured_failures = (
    (
        (df["Machine failure"] == 1)
        &
        (df["Anomaly"] == 1)
    )
).sum()

capture_rate = (
    captured_failures / total_failures
)

print("Total actual failures:", total_failures)

print(
    "Failures detected as anomalies:",
    captured_failures
)

print(
    "Failure capture rate:",
    round(capture_rate, 4)
)

print(
    "Failure capture percentage:",
    round(capture_rate * 100, 2),
    "%"
)

Total actual failures: 339
Failures detected as anomalies: 61
Failure capture rate: 0.1799
Failure capture percentage: 17.99 %


test contamination values

In [10]:
contamination_values = [
    0.02,
    0.03,
    0.04,
    0.05,
    0.06,
    0.08,
    0.10,
    0.12,
    0.15
]

results = []

y_true = df["Machine failure"].values

for contamination in contamination_values:

    model = IsolationForest(
        n_estimators=300,
        contamination=contamination,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_anomaly_scaled)

    pred = model.predict(X_anomaly_scaled)

    y_pred = np.where(
        pred == -1,
        1,
        0
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    results.append({
        "Contamination": contamination,
        "Anomalies": y_pred.sum(),
        "Precision": precision,
        "Failure Capture (Recall)": recall,
        "F1": f1
    })

contamination_results = pd.DataFrame(results)

contamination_results.round(4)

,Contamination,Anomalies,Precision,Failure Capture (Recall),F1
0,0.02,200,0.2150,0.1268,0.1596
1,0.03,300,0.1633,0.1445,0.1534
2,0.04,400,0.1525,0.1799,0.1651
3,0.05,500,0.1400,0.2065,0.1669
4,0.06,600,0.1400,0.2478,0.1789
5,0.08,800,0.1325,0.3127,0.1861
6,0.10,1000,0.1200,0.3540,0.1792
7,0.12,1200,0.1167,0.4130,0.1819
8,0.15,1500,0.1120,0.4956,0.1827


Find best F1 operating point

In [11]:
best_anomaly_row = contamination_results.loc[
    contamination_results["F1"].idxmax()
]

print("BEST F1 CONFIGURATION")
print("---------------------")

print(best_anomaly_row)

BEST F1 CONFIGURATION
---------------------
Contamination                 0.080000
Anomalies                   800.000000
Precision                     0.132500
Failure Capture (Recall)      0.312684
F1                            0.186128
Name: 5, dtype: float64


Check anomaly score discrimination

In [12]:
score_summary = (
    df.groupby("Machine failure")["Anomaly Score"]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max"
    ])
)

score_summary

,count,mean,median,std,min,max
Machine failure,,,,,,
0,9661,-0.112141,-0.123881,0.046779,-0.180873,0.125664
1,339,-0.049786,-0.061632,0.056980,-0.163158,0.117500


ROC-AUC and PR-AUC of anomaly score

In [13]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

anomaly_roc_auc = roc_auc_score(
    df["Machine failure"],
    df["Anomaly Score"]
)

anomaly_pr_auc = average_precision_score(
    df["Machine failure"],
    df["Anomaly Score"]
)

print("Anomaly Score ROC-AUC:", anomaly_roc_auc)
print("Anomaly Score PR-AUC :", anomaly_pr_auc)

print(
    "Random PR baseline  :",
    df["Machine failure"].mean()
)

Anomaly Score ROC-AUC: 0.8201789941555608
Anomaly Score PR-AUC : 0.1461707290974864
Random PR baseline  : 0.0339


Train final Isolation Forest

In [14]:
final_anomaly_model = IsolationForest(
    n_estimators=300,
    contamination=0.08,
    random_state=42,
    n_jobs=-1
)

final_anomaly_model.fit(
    X_anomaly_scaled
)

print("Final Isolation Forest trained!")

Final Isolation Forest trained!


Final anomaly predictions

In [15]:
final_raw_pred = final_anomaly_model.predict(
    X_anomaly_scaled
)

final_anomaly_score = (
    -final_anomaly_model.decision_function(
        X_anomaly_scaled
    )
)

df["Final Anomaly"] = np.where(
    final_raw_pred == -1,
    1,
    0
)

df["Final Anomaly Score"] = final_anomaly_score

print(df["Final Anomaly"].value_counts())

print(
    "Anomaly percentage:",
    round(df["Final Anomaly"].mean() * 100, 2),
    "%"
)

Final Anomaly
0    9200
1     800
Name: count, dtype: int64
Anomaly percentage: 8.0 %


Save model AND scaler

In [16]:
import joblib

joblib.dump(
    final_anomaly_model,
    "../models/anomaly_isolation_forest.joblib"
)

joblib.dump(
    scaler,
    "../models/anomaly_scaler.joblib"
)

print("Anomaly model saved!")
print("Anomaly scaler saved!")

Anomaly model saved!
Anomaly scaler saved!


Save configuration

In [17]:
import json

anomaly_config = {
    "model_name": "Isolation Forest",
    "dataset": "AI4I 2020",

    "features": anomaly_features,

    "n_estimators": 300,
    "contamination": 0.08,
    "random_state": 42,

    "evaluation": {
        "precision_vs_failure": 0.1325,
        "failure_capture_recall": 0.312684,
        "f1_vs_failure": 0.186128,
        "roc_auc_score": 0.820179,
        "pr_auc_score": 0.146171,
        "random_pr_baseline": 0.0339
    },

    "interpretation": {
        "prediction_1": "Normal",
        "prediction_-1": "Anomaly",
        "higher_custom_anomaly_score": "More anomalous"
    }
}

with open(
    "../models/anomaly_config.json",
    "w"
) as file:

    json.dump(
        anomaly_config,
        file,
        indent=4
    )

print("Anomaly configuration saved!")

Anomaly configuration saved!


Reload and verify

In [18]:
loaded_anomaly_model = joblib.load(
    "../models/anomaly_isolation_forest.joblib"
)

loaded_anomaly_scaler = joblib.load(
    "../models/anomaly_scaler.joblib"
)

X_check = loaded_anomaly_scaler.transform(
    X_anomaly
)

original_pred = final_anomaly_model.predict(
    X_anomaly_scaled
)

loaded_pred = loaded_anomaly_model.predict(
    X_check
)

print(
    "Predictions identical:",
    np.array_equal(
        original_pred,
        loaded_pred
    )
)

Predictions identical: True
